# LeetCode 254 — Factor Combinations bằng Python

Link đề: https://leetcode.com/problems/factor-combinations/ · Độ khó: **Medium** (Premium) · Chủ đề: Backtracking, Math, Recursion (phân tích thành tích các ước)

Skeleton LeetCode:

```python
class Solution:
    def getFactors(self, n: int) -> List[List[int]]:
        ...
```

Ghi chú: đề bài viết theo hiểu biết về bài (bài Premium), nên đối chiếu lại nếu có tài khoản Premium; ví dụ và constraints lấy từ file đề có sẵn. Mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với brute force độc lập (tách đôi `n = a × b` rồi đệ quy, gom tập các bộ đã sắp).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: liệt kê **mọi cách viết `n` thành tích các thừa số** trong khoảng `[2, n - 1]` (không tính cách viết `n = n` và không tính hoán vị là cách khác).
2. Phân biệt **bộ thừa số (không quan tâm thứ tự)** với **dãy**: `[2, 6]` và `[6, 2]` là **một** cách; ta quy ước **thừa số không giảm** để mỗi cách chỉ xuất hiện **một lần**.
3. Đi từ cách **thử mọi ước từ `start` đến `n - 1`** → **chỉ thử tới `√m`** (backtracking có cắt tỉa) → **chỉ thử trên danh sách ước của `n`** → **đệ quy có nhớ (memoization)** → **DFS bằng stack tường minh**.
4. Hiểu khung **"chọn một ước `d ≥ start`, chia `m` cho `d`, đệ quy"** và vì sao có thể ghi nhận `path + [m // d]` **ngay** khi chọn `d`.
5. Tránh các lỗi kinh điển: sinh **hoán vị trùng**, đưa **chính `n`** (hoặc `1`) vào đáp án, **quên thừa số cuối**, không **cắt tỉa `d*d ≤ m`**, quên **sao chép** `path`.

## 2. Đề bài & ví dụ

Số có thể xem là **tích các thừa số** của nó. Ví dụ `8 = 2 × 2 × 2 = 2 × 4`.

Cho số nguyên `n`, trả về **mọi tổ hợp thừa số** có thể có của `n` (thứ tự bất kỳ).

Lưu ý: các thừa số phải nằm trong khoảng `[2, n - 1]`.

**Ví dụ 1**

- Input: `n = 1`
- Output: `[]`

**Ví dụ 2**

- Input: `n = 12`
- Output: `[[2,6],[3,4],[2,2,3]]`

**Ví dụ 3**

- Input: `n = 37`
- Output: `[]`
- Giải thích: `37` là số nguyên tố nên chỉ viết được `37 = 37` (không hợp lệ vì thừa số phải `<= n - 1`).

**Ràng buộc:** `1 <= n <= 10^7`.

In [1]:
from typing import List
from functools import lru_cache
import random
import time

examples = [
    (1, []),
    (12, [[2, 6], [3, 4], [2, 2, 3]]),
    (37, []),
]


def normalize(result):
    """Chuẩn hóa để so sánh: mỗi bộ sắp tăng dần, rồi sắp toàn bộ danh sách (đề cho phép thứ tự bất kỳ)."""
    return sorted(sorted(c) for c in result)


for n, expected in examples:
    print(f"n={n} -> kỳ vọng {normalize(expected)}")

n=1 -> kỳ vọng []
n=12 -> kỳ vọng [[2, 2, 3], [2, 6], [3, 4]]
n=37 -> kỳ vọng []


## 3. Kiến thức nền

### 3.1. Ước (divisor), bội, `%` và `//`

`d` là **ước** của `m` nếu `m % d == 0` (chia hết, dư `0`). Khi đó `m // d` là **thương nguyên** và `d × (m // d) == m`. Ví dụ `12 % 3 == 0`, `12 // 3 == 4`.
#### Code demo

In [2]:
for d in range(1, 13):
    if 12 % d == 0:
        print(f"{d} là ước của 12: 12 = {d} × {12 // d}")
print(12 % 5, 12 // 5)       # 2 2: 5 không là ước

1 là ước của 12: 12 = 1 × 12
2 là ước của 12: 12 = 2 × 6
3 là ước của 12: 12 = 3 × 4
4 là ước của 12: 12 = 4 × 3
6 là ước của 12: 12 = 6 × 2
12 là ước của 12: 12 = 12 × 1
2 2


### 3.2. Vì sao chỉ cần thử đến `√m`

Nếu `m = a × b` với `a ≤ b` thì `a² ≤ a × b = m`, tức `a ≤ √m`. Do đó mọi cách tách `m` thành **hai** thừa số `a ≤ b` đều có thừa số nhỏ `a` thỏa `a * a ≤ m`. Điều kiện `d * d <= m` thay cho `d <= sqrt(m)` (tránh số thực).
#### Code demo

In [3]:
m = 36
pairs = [(a, m // a) for a in range(2, m) if a * a <= m and m % a == 0]
print(pairs)        # [(2, 18), (3, 12), (4, 9), (6, 6)]: mỗi cặp a <= b xuất hiện đúng một lần

[(2, 18), (3, 12), (4, 9), (6, 6)]


### 3.3. Tổ hợp (bộ) và quy ước thừa số không giảm

`2 × 6` và `6 × 2` là cùng một cách phân tích. Để mỗi cách chỉ xuất hiện **một lần**, ta chỉ xét các bộ **không giảm**: `[2, 2, 3]` chứ không `[2, 3, 2]`. Khi đệ quy, truyền `start` = thừa số vừa chọn và chỉ chọn thừa số **kế tiếp `>= start`**.
#### Code demo

In [4]:
from itertools import permutations
print(sorted(set(permutations([2, 2, 3]))))     # 3 hoán vị khác nhau của cùng một cách phân tích
print("quy ước không giảm chỉ giữ:", [2, 2, 3])

[(2, 2, 3), (2, 3, 2), (3, 2, 2)]
quy ước không giảm chỉ giữ: [2, 2, 3]


### 3.4. Backtracking: chọn → đệ quy → bỏ chọn

Ta dùng danh sách `path` chứa các thừa số đã chọn. `path.append(d)` rồi `dfs(...)` rồi `path.pop()`. Khi cần **lưu** một đáp án, ghi `path + [m // d]` (tạo danh sách **mới**) để khỏi bị biến đổi về sau. `path + [x]` khác `path.append(x)`: cái trước tạo bản mới, cái sau sửa tại chỗ.
#### Code demo

In [5]:
path = [2, 3]
saved = path + [7]          # bản mới
print(saved, path)          # [2, 3, 7] [2, 3]
alias = path
alias.append(9)
print(path)                 # [2, 3, 9]: alias là CÙNG danh sách

[2, 3, 7] [2, 3]
[2, 3, 9]


### 3.5. Phân tích thành thừa số: cấu trúc "chọn ước nhỏ nhất, phần còn lại tự do"

Một cách phân tích không giảm `[f1, f2, ..., fk]` có `f1 ≤ f2 ≤ ... ≤ fk` và tích `= n`. **Thừa số nhỏ nhất `f1`** là một ước của `n` thỏa `f1 ≤ √n` (nếu `k ≥ 2`). Sau khi chọn `f1`, phần còn lại `n / f1` lại được phân tích **theo cùng cách**, với mọi thừa số `≥ f1`. Đây là **cấu trúc đệ quy**: `ways(m, start)` = `{[m]}` (nếu `m ≥ start`) ∪ mọi `{[d] + w : d ≥ start, d | m, d*d ≤ m, w ∈ ways(m/d, d)}`.
#### Code demo

In [6]:
def ways_demo(m, start):
    res = [[m]] if m >= start else []
    d = start
    while d * d <= m:
        if m % d == 0:
            res += [[d] + w for w in ways_demo(m // d, d)]
        d += 1
    return res


print(ways_demo(12, 2))     # [[12], [2, 6], [2, 2, 3], [3, 4]]  (cả [12] chưa bị loại)

[[12], [2, 6], [2, 2, 3], [3, 4]]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tính tay `n = 12`. Chọn thừa số nhỏ nhất `d` không giảm:

- `d = 2`: `12 = 2 × 6` → ghi `[2, 6]`; tiếp tục phân tích `6` với thừa số `≥ 2`:
  - `d = 2`: `6 = 2 × 3` → ghi `[2, 2, 3]`; phân tích `3` với thừa số `≥ 2`: `d*d = 4 > 3`, dừng.
  - `d = 3`: `3*3 = 9 > 6`, dừng.
- `d = 3`: `12 = 3 × 4` → ghi `[3, 4]`; phân tích `4` với thừa số `≥ 3`: `3*3 = 9 > 4`, dừng.
- `d = 4`: `4*4 = 16 > 12`, dừng.

Kết quả `[2,6]`, `[2,2,3]`, `[3,4]` — đủ 3 cách như ví dụ 2. Chú ý: **không ghi `[12]`** vì thừa số phải `<= n-1`; và ta **ghi nhận ngay** khi tách một ước ra (cùng với phần còn lại làm thừa số cuối), vì phần còn lại `m/d ≥ d` luôn hợp lệ.

## 5. Cách 1 — Brute Force / baseline: thử mọi ước từ `start` đến `m - 1`

**Ý tưởng.** Hàm `dfs(m, start)`: duyệt **mọi** `d` từ `start` đến `m - 1` (không cắt tỉa); nếu `d` là ước của `m` **và** `m // d >= d`, ghi `path + [d, m // d]` rồi đệ quy `dfs(m // d, d)` sau khi đẩy `d` vào `path`. (Điều kiện `m // d >= d` thay cho cắt tỉa `d*d <= m`, nhưng vẫn quét hết các `d`.)
#### Code demo

In [7]:
def factors_bruteforce(n):
    ans, path = [], []

    def dfs(m, start):
        for d in range(start, m):                       # quét mọi d, không cắt tỉa
            if m % d == 0 and m // d >= d:
                ans.append(path + [d, m // d])           # ghi nhận: các thừa số đã chọn + d + phần còn lại
                path.append(d)
                dfs(m // d, d)
                path.pop()

    dfs(n, 2)
    return ans


for n, expected in examples:
    print(n, "->", normalize(factors_bruteforce(n)), "(kỳ vọng", normalize(expected), ")")

1 -> [] (kỳ vọng [] )
12 -> [[2, 2, 3], [2, 6], [3, 4]] (kỳ vọng [[2, 2, 3], [2, 6], [3, 4]] )
37 -> [] (kỳ vọng [] )


**Độ phức tạp (Cách 1).** Mỗi lời gọi quét tới `m` giá trị `d`: tầng đầu `O(n)`; với `n = 10^7` là `10^7` phép chỉ riêng tầng đầu — **quá chậm**. **Space `O(log n)`** (độ sâu đệ quy tối đa `log₂ n`).

**Nút thắt.** Phần lớn `d` lớn hơn `√m` đều bị loại bởi `m // d >= d`; quét chúng là lãng phí. Ta chỉ cần quét `d` với `d * d <= m`.

## 6. Ý tưởng thuật toán / bước đột phá

**Cắt tỉa theo `√m`:** vì `m // d >= d ⟺ d * d <= m` (với `d` là ước), vòng lặp chỉ cần chạy `while d * d <= m`. Tầng đầu chỉ còn `~ √n ≈ 3162` bước cho `n = 10^7`, và các tầng sâu hơn còn nhỏ hơn nữa (vì `m` giảm).

**Quan sát thêm:** thay vì thử mọi `d`, chỉ thử các **ước của `n`** (số ước của `n ≤ 10^7` không quá `448`); phần còn lại `m` luôn là một ước của `n`, nên `d` chạy trên danh sách ước đã sắp.

**Ghi nhận ngay:** khi chọn `d` (ước, `d*d ≤ m`), ta biết `m // d ≥ d` nên `path + [d, m // d]` đã là một cách phân tích **hợp lệ** (mọi thừa số `≥ 2`, không giảm, tích `= n`). Sau đó chỉ cần xét tiếp việc **tách nhỏ hơn** `m // d`.

## 7. Các cách tiếp theo

### Cách 2 — Backtracking có cắt tỉa `d * d ≤ m` (cách nộp chuẩn)

**Giải thích.** `dfs(m, i)`: nếu `path` không rỗng thì ghi `path + [m]` (phần còn lại `m` làm thừa số cuối); rồi với `j` từ `i` mà `j * j <= m`: nếu `j` là ước, đẩy `j` vào `path`, gọi `dfs(m // j, j)`, rồi bỏ `j`.
#### Code demo

In [8]:
def factors_backtrack(n):
    ans, path = [], []

    def dfs(m, i):
        if path:
            ans.append(path + [m])                        # thừa số cuối là phần còn lại m (luôn >= phần tử cuối của path)
        j = i
        while j * j <= m:                                  # cắt tỉa theo căn bậc hai
            if m % j == 0:
                path.append(j)
                dfs(m // j, j)                             # từ nay chỉ chọn thừa số >= j (không giảm)
                path.pop()
            j += 1

    dfs(n, 2)
    return ans


for n, expected in examples:
    print(n, "->", normalize(factors_backtrack(n)), "(kỳ vọng", normalize(expected), ")")

1 -> [] (kỳ vọng [] )
12 -> [[2, 2, 3], [2, 6], [3, 4]] (kỳ vọng [[2, 2, 3], [2, 6], [3, 4]] )
37 -> [] (kỳ vọng [] )


In [9]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def getFactors(self, n: int) -> List[List[int]]:
        def dfs(n: int, i: int) -> None:
            if t:
                ans.append(t + [n])
            j = i
            while j * j <= n:
                if n % j == 0:
                    t.append(j)
                    dfs(n // j, j)
                    t.pop()
                j += 1

        t = []
        ans = []
        dfs(n, 2)
        return ans


print(normalize(Solution().getFactors(12)))

[[2, 2, 3], [2, 6], [3, 4]]


**Độ phức tạp Cách 2:** gọi `K` là số kết quả. Mỗi lời gọi tốn `O(√m)` để quét; số lời gọi bằng số nút của cây đệ quy `≈ K + (số nhánh cụt)`. Tầng đầu `O(√n)`. **Time `O(√n + K·(độ dài + căn các tầng sâu))`**, thực tế rất nhỏ; **Space `O(log n)`** cho `path`/ngăn xếp (độ sâu tối đa `log₂ n ≈ 23`), ngoài kết quả.

### Cách 3 — Chỉ duyệt trên danh sách ước của `n`

**Giải thích.** Tìm mọi ước `d` của `n` với `2 ≤ d ≤ n/2` (một lần, `O(√n)`), sắp tăng dần: `divs`. Ở `dfs(m, k)`, chỉ thử `divs[k:]` theo thứ tự, dừng khi `d*d > m`. Chỉ phải thử vài trăm ước thay vì cả đoạn `[start, √m]`.
#### Code demo

In [10]:
def divisors_of(n):
    small, large = [], []
    d = 1
    while d * d <= n:
        if n % d == 0:
            small.append(d)
            if d != n // d:
                large.append(n // d)
        d += 1
    return small + large[::-1]                              # đã sắp tăng dần


def factors_divisors(n):
    divs = [d for d in divisors_of(n) if 2 <= d <= n // 2]
    ans, path = [], []

    def dfs(m, k):
        for idx in range(k, len(divs)):
            d = divs[idx]
            if d * d > m:
                break
            if m % d == 0:
                path.append(d)
                ans.append(path + [m // d])
                dfs(m // d, idx)
                path.pop()

    dfs(n, 0)
    return ans


for n, expected in examples:
    print(n, "->", normalize(factors_divisors(n)), "(kỳ vọng", normalize(expected), ")")

1 -> [] (kỳ vọng [] )
12 -> [[2, 2, 3], [2, 6], [3, 4]] (kỳ vọng [[2, 2, 3], [2, 6], [3, 4]] )
37 -> [] (kỳ vọng [] )


**Độ phức tạp Cách 3:** tìm ước `O(√n)`; mỗi lời gọi quét tối đa `D` ước với `D ≤ 448` (cho `n ≤ 10^7`): **Time `O(√n + K·D)`** trong worst case, **Space `O(D + log n)`**. Hơi nhanh hơn Cách 2 khi `n` có ít ước nhưng lớn.

### Cách 4 — Đệ quy có nhớ (memoization)

**Giải thích.** Định nghĩa `f(m, lo)` = **mọi cách** phân tích `m` thành các thừa số **không giảm, mỗi thừa số `≥ lo`** (kể cả cách chỉ có một thừa số `(m,)` nếu `m ≥ lo`). Khi nhiều nhánh gặp cùng `(m, lo)`, kết quả được **dùng lại**. Đáp án là `f(n, 2)` bỏ đi cách `(n,)`.
#### Code demo

In [11]:
def factors_memo(n):
    @lru_cache(maxsize=None)
    def f(m, lo):
        res = []
        if m >= lo:
            res.append((m,))
        d = lo
        while d * d <= m:
            if m % d == 0:
                for rest in f(m // d, d):
                    res.append((d,) + rest)
            d += 1
        return tuple(res)

    if n < 4:
        return []
    return [list(t) for t in f(n, 2) if t != (n,)]


for n, expected in examples:
    print(n, "->", normalize(factors_memo(n)), "(kỳ vọng", normalize(expected), ")")

1 -> [] (kỳ vọng [] )
12 -> [[2, 2, 3], [2, 6], [3, 4]] (kỳ vọng [[2, 2, 3], [2, 6], [3, 4]] )
37 -> [] (kỳ vọng [] )


**Độ phức tạp Cách 4:** số trạng thái `(m, lo)` bị chặn bởi `(số ước)²`; mỗi trạng thái nối `O(K_m)` kết quả: **Time** phụ thuộc đầu ra nhưng thường ít lặp lại hơn; **Space `O(tổng kích thước các kết quả đã nhớ)`** — lớn hơn Cách 2. Lợi ích rõ khi `n` có nhiều ước chung cấu trúc (ví dụ `n = 2^a·3^b`).

### Cách 5 — DFS bằng stack tường minh (không đệ quy)

**Giải thích.** Mỗi phần tử stack là `(m, i, path)` với `path` là bộ các thừa số đã chọn. Lấy ra, ghi nhận nếu `path` không rỗng, rồi đẩy các nhánh con `(m // j, j, path + (j,))` cho mọi ước `j` hợp lệ.
#### Code demo

In [12]:
def factors_stack(n):
    ans = []
    stack = [(n, 2, ())]
    while stack:
        m, i, path = stack.pop()
        if path:
            ans.append(list(path) + [m])
        j = i
        while j * j <= m:
            if m % j == 0:
                stack.append((m // j, j, path + (j,)))
            j += 1
    return ans


for n, expected in examples:
    print(n, "->", normalize(factors_stack(n)), "(kỳ vọng", normalize(expected), ")")

1 -> [] (kỳ vọng [] )
12 -> [[2, 2, 3], [2, 6], [3, 4]] (kỳ vọng [[2, 2, 3], [2, 6], [3, 4]] )
37 -> [] (kỳ vọng [] )


**Độ phức tạp Cách 5:** **Time** như Cách 2; **Space `O(độ sâu × số nhánh chưa xử lý)`** nhưng vẫn nhỏ vì mỗi tầng ≤ `√m` nhánh và độ sâu ≤ `log n`; mỗi `path` là một tuple riêng (an toàn khi lưu).

> Cách 2 là bản nộp phổ biến, ngắn gọn. Cách 3 giúp thấy cấu trúc ước. Cách 4 là cầu nối sang bài đếm số cách (DP).

## 8. TRACE từng bước

Ta in từng lời gọi `dfs(m, i)` của Cách 2 cho `n = 12` (thụt lề theo độ sâu), kèm lý do dừng.
#### Code demo

In [13]:
def trace_backtrack(n):
    ans, path = [], []

    def dfs(m, i, depth=0):
        pad = "  " * depth
        print(f"{pad}dfs(m={m}, i={i}), path={path}")
        if path:
            ans.append(path + [m])
            print(f"{pad}  => ghi nhận {path + [m]}")
        j = i
        while j * j <= m:
            if m % j == 0:
                print(f"{pad}  j={j}: {j}*{j}={j * j} <= {m} và {m} % {j} == 0 -> chọn {j}")
                path.append(j)
                dfs(m // j, j, depth + 1)
                path.pop()
            j += 1
        print(f"{pad}  (dừng: j={j}, {j}*{j}={j * j} > {m})")

    dfs(n, 2)
    print("kết quả:", ans)
    return ans


trace_backtrack(12)

dfs(m=12, i=2), path=[]
  j=2: 2*2=4 <= 12 và 12 % 2 == 0 -> chọn 2
  dfs(m=6, i=2), path=[2]
    => ghi nhận [2, 6]
    j=2: 2*2=4 <= 6 và 6 % 2 == 0 -> chọn 2
    dfs(m=3, i=2), path=[2, 2]
      => ghi nhận [2, 2, 3]
      (dừng: j=2, 2*2=4 > 3)
    (dừng: j=3, 3*3=9 > 6)
  j=3: 3*3=9 <= 12 và 12 % 3 == 0 -> chọn 3
  dfs(m=4, i=3), path=[3]
    => ghi nhận [3, 4]
    (dừng: j=3, 3*3=9 > 4)
  (dừng: j=4, 4*4=16 > 12)
kết quả: [[2, 6], [2, 2, 3], [3, 4]]


[[2, 6], [2, 2, 3], [3, 4]]

Bảng tóm tắt (`n = 12`):

| `path` | `m` còn lại | thử `j` | kết quả ghi nhận |
|---|---|---|---|
| `[]` | `12` | `2` (ước), `3` (ước), `4`: `16 > 12` dừng | (chưa ghi, `path` rỗng) |
| `[2]` | `6` | `2` (ước), `3`: `9 > 6` dừng | `[2, 6]` |
| `[2, 2]` | `3` | `2`: `4 > 3` dừng | `[2, 2, 3]` |
| `[3]` | `4` | `3`: `9 > 4` dừng | `[3, 4]` |

Chú ý lời gọi đầu tiên (`path` rỗng) **không** ghi `[12]`: đó chính là lý do `if path:`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (backtracking có cắt tỉa). Đúng `class Solution`, đúng `getFactors(self, n: int) -> List[List[int]]`.

In [14]:
class Solution:
    def getFactors(self, n: int) -> List[List[int]]:
        def dfs(m: int, i: int) -> None:
            if path:                                 # path không rỗng: m là thừa số cuối hợp lệ
                ans.append(path + [m])               # path + [m] tạo danh sách mới (không bị alias)
            j = i
            while j * j <= m:                        # cắt tỉa: chỉ thử ước nhỏ nhất có thể
                if m % j == 0:
                    path.append(j)
                    dfs(m // j, j)                   # truyền j để thừa số không giảm
                    path.pop()
                j += 1

        path: List[int] = []
        ans: List[List[int]] = []
        dfs(n, 2)
        return ans


sol = Solution()
for n, expected in examples:
    assert normalize(sol.getFactors(n)) == normalize(expected)
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Mỗi đáp án sinh đúng một lần:** mọi cách phân tích không giảm `[f1 ≤ f2 ≤ ... ≤ fk]` (`k ≥ 2`) tương ứng **một đường duy nhất** `f1, f2, ..., f_{k-1}` trong cây đệ quy (ta đẩy các thừa số nhỏ dần lên trước, còn `fk` là phần còn lại `m`). Vì `start` tăng không giảm nên không có hoán vị trùng.
- **Hợp lệ:** khi chọn `j` với `j*j ≤ m` và `j | m`, phần còn lại `m // j ≥ j ≥ 2` nên mọi thừa số trong `path + [m // j]` thuộc `[2, n-1]` và không giảm; tích `= n`. Thừa số `n` không bao giờ xuất hiện đơn lẻ (path rỗng thì không ghi).
- **Đầy đủ:** với bất kỳ cách phân tích không giảm có `k ≥ 2` thừa số, `f1` thỏa `f1² ≤ f1·f2 ≤ n`, và (quy nạp) `f2 .. fk` là một cách phân tích của `n/f1` với thừa số `≥ f1` — thuộc cây con tương ứng.
- **Kết thúc:** `m` giảm ít nhất một nửa mỗi lần đệ quy (`m // j ≤ m / 2`), độ sâu `≤ log₂ n`.
- **Cắt tỉa `j*j ≤ m`:** các `j` bị bỏ không thể là thừa số nhỏ nhất của một cách phân tích có ít nhất hai thừa số của `m`.

## 11. Phân tích Time / Space Complexity

Gọi `K` là số kết quả, `L ≤ log₂ n ≈ 23` là độ dài tối đa của một kết quả.

| Cách | Time | Space (ngoài kết quả) |
|---|---|---|
| 1. Quét `d` tới `m - 1` | `O(n + ...)` (tầng đầu `O(n)`) | `O(L)` |
| 2. Backtracking `d*d ≤ m` | `O(√n + K·L·√m_trung_bình)` thực tế rất nhỏ | `O(L)` |
| 3. Chỉ trên danh sách ước | `O(√n + K·D)`, `D ≤ 448` | `O(D + L)` |
| 4. Có nhớ | phụ thuộc đầu ra, ít lặp | `O(tổng kết quả nhớ)` |
| 5. Stack tường minh | như Cách 2 | `O(L·√n)` tệ nhất |

Với `n = 10^7`, tầng đầu chỉ `~3162` bước và số kết quả `K` cỡ vài nghìn — chạy trong vài chục mili giây. Kích thước **đầu ra** `K·L` là một cận dưới tự nhiên. Không có khác biệt average/worst đáng kể (thời gian phụ thuộc vào cấu trúc ước của `n`: lũy thừa của số nhỏ như `2^23` cho nhiều kết quả).

## 12. Bộ test

`oracle` là brute force **độc lập**: tập `S(m)` = mọi cách viết `m` thành tích các thừa số `≥ 2` (kể cả `(m,)`), tính đệ quy bằng cách **tách đôi** `m = a × b` (với `2 ≤ a ≤ b`) và ghép các cách của `a` với các cách của `b` rồi **sắp tăng dần** và gom vào tập. Cách này không dùng `start` hay thứ tự không giảm khi duyệt.
#### Code demo

In [15]:
@lru_cache(maxsize=None)
def multiplicative_partitions(m):
    """Mọi bộ thừa số >= 2 (đã sắp tăng dần) có tích m, gồm cả (m,); tính bằng cách tách đôi m = a * b."""
    result = {(m,)}
    a = 2
    while a * a <= m:
        if m % a == 0:
            b = m // a
            for x in multiplicative_partitions(a):
                for y in multiplicative_partitions(b):
                    result.add(tuple(sorted(x + y)))
        a += 1
    return frozenset(result)


def oracle(n):
    if n < 4:
        return []
    return sorted(list(t) for t in multiplicative_partitions(n) if t != (n,))


impls = {
    "bruteforce": factors_bruteforce,
    "backtrack": factors_backtrack,
    "divisors": factors_divisors,
    "memo": factors_memo,
    "stack": factors_stack,
    "submit": sol.getFactors,
}

for n, expected in examples:
    assert oracle(n) == normalize(expected)
    for key, fn in impls.items():
        assert normalize(fn(n)) == normalize(expected), (key, n, normalize(fn(n)), normalize(expected))
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên
#### Code demo

In [16]:
edge_cases = [
    (1, [], "n = 1"),
    (2, [], "số nguyên tố nhỏ nhất"),
    (3, [], "số nguyên tố"),
    (4, [[2, 2]], "bình phương số nguyên tố"),
    (6, [[2, 3]], "tích hai số nguyên tố khác nhau"),
    (8, [[2, 4], [2, 2, 2]], "lũy thừa của 2"),
    (16, [[2, 8], [4, 4], [2, 2, 4], [2, 2, 2, 2]], "2^4: có 4 cách"),
    (12, [[2, 6], [3, 4], [2, 2, 3]], "ví dụ của đề"),
    (30, [[2, 15], [3, 10], [5, 6], [2, 3, 5]], "tích ba số nguyên tố"),
    (36, [[2, 18], [3, 12], [4, 9], [6, 6], [2, 2, 9], [2, 3, 6], [3, 3, 4], [2, 2, 3, 3]], "bình phương hoàn hảo"),
    (37, [], "số nguyên tố lớn hơn"),
]
for n, expected, label in edge_cases:
    assert oracle(n) == normalize(expected), (label, oracle(n), normalize(expected))
    for key, fn in impls.items():
        assert normalize(fn(n)) == normalize(expected), (key, label, n, normalize(fn(n)), normalize(expected))
    print("PASS", label)

PASS n = 1
PASS số nguyên tố nhỏ nhất
PASS số nguyên tố
PASS bình phương số nguyên tố
PASS tích hai số nguyên tố khác nhau
PASS lũy thừa của 2
PASS 2^4: có 4 cách
PASS ví dụ của đề
PASS tích ba số nguyên tố
PASS bình phương hoàn hảo
PASS số nguyên tố lớn hơn


### 12.2. Tính chất và phản ví dụ

- Mỗi bộ trong đáp án: **tích bằng `n`**, mọi thừa số trong `[2, n - 1]`, **không giảm**, **độ dài `≥ 2`**; các bộ **đôi một khác nhau**.
- Số kết quả `= (số cách phân tích có thứ tự bỏ qua) - 1`: với `n = 2^k` là `p(k) - 1` (`p` là hàm phân hoạch số nguyên: `p(1..6) = 1, 2, 3, 5, 7, 11`).
- `n` nguyên tố ⇒ không có kết quả.
- **Phản ví dụ** cho lỗi sinh hoán vị: bản thử mọi ước không giữ thứ tự không giảm cho `n = 12` ra `[2,6]`, `[6,2]`, `[3,4]`, `[4,3]`, ...
#### Code demo

In [17]:
for n in range(1, 400):
    res = sol.getFactors(n)
    seen = set()
    for c in res:
        prod = 1
        for x in c:
            prod *= x
        assert prod == n and len(c) >= 2 and all(2 <= x <= n - 1 for x in c)
        assert all(c[i] <= c[i + 1] for i in range(len(c) - 1))
        seen.add(tuple(c))
    assert len(seen) == len(res)
    assert len(res) == len(oracle(n))
print("PASS tính chất (tích = n, trong [2, n-1], không giảm, khác nhau, số lượng) cho n = 1..399")

partition_counts = {1: 1, 2: 2, 3: 3, 4: 5, 5: 7, 6: 11}
for k, p in partition_counts.items():
    assert len(sol.getFactors(2 ** k)) == p - 1
print("PASS số cách của 2^k bằng p(k) - 1")


def sai_hoan_vi(n):
    ans, path = [], []

    def dfs(m):
        if path:
            ans.append(path + [m])
        for j in range(2, m):                              # SAI: luôn bắt đầu từ 2, không giữ thứ tự không giảm
            if m % j == 0:
                path.append(j)
                dfs(m // j)
                path.pop()

    dfs(n)
    return ans


print("phản ví dụ n=12: đúng", sorted(sol.getFactors(12)), "| hoán vị:", sorted(sai_hoan_vi(12)))
assert len(sai_hoan_vi(12)) > len(sol.getFactors(12))

PASS tính chất (tích = n, trong [2, n-1], không giảm, khác nhau, số lượng) cho n = 1..399
PASS số cách của 2^k bằng p(k) - 1
phản ví dụ n=12: đúng [[2, 2, 3], [2, 6], [3, 4]] | hoán vị: [[2, 2, 3], [2, 3, 2], [2, 6], [3, 2, 2], [3, 4], [4, 3], [6, 2]]


### 12.3. Đối chiếu ngẫu nhiên (có seed) và `n` lớn
#### Code demo

In [18]:
random.seed(254)
for n in list(range(1, 301)) + random.sample(range(301, 3000), 150):
    truth = oracle(n)
    for key, fn in impls.items():
        if key == "bruteforce" and n > 600:
            continue                                        # bruteforce chậm hơn nhưng đã phủ tới 600
        assert normalize(fn(n)) == truth, (key, n)
print("PASS n = 1..300 và 150 giá trị ngẫu nhiên tới 3000: mọi cách khớp oracle")

# n lớn: so các cách nhanh với nhau (oracle tách đôi còn dùng được tới ~10^5)
big_values = [2 ** 23, 10 ** 7, 9_699_690, 9_973 * 1000, 8_388_607, 2 ** 20 * 9, 5_040 * 1_000]
for n in big_values:
    ref = None
    row = []
    for key in ("backtrack", "divisors", "stack", "submit"):
        t0 = time.perf_counter()
        res = normalize(impls[key](n))
        dt = (time.perf_counter() - t0) * 1000
        ref = res if ref is None else ref
        assert res == ref, (key, n)
        row.append(f"{key}={dt:6.1f}ms")
    print(f"n={n:>9}: {len(ref):>6} cách | " + " ".join(row))
print("PASS n lớn: các cách khớp nhau")

for n in (10 ** 5, 9 * 10 ** 4 + 10):
    assert normalize(factors_memo(n)) == oracle(n) == normalize(sol.getFactors(n))
print("PASS n ~ 10^5: Cách có nhớ, oracle, nộp khớp nhau")

PASS n = 1..300 và 150 giá trị ngẫu nhiên tới 3000: mọi cách khớp oracle
n=  8388608:   1254 cách | backtrack=   3.2ms divisors=   1.2ms stack=   3.5ms submit=   3.0ms
n= 10000000:   2997 cách | backtrack=   6.7ms divisors=   2.5ms stack=   6.5ms submit=   6.2ms
n=  9699690:   4139 cách | backtrack=   7.9ms divisors=   3.1ms stack=   7.2ms submit=   7.9ms
n=  9973000:    108 cách | backtrack=   2.0ms divisors=   0.3ms stack=   1.9ms submit=   1.9ms
n=  8388607:      1 cách | backtrack=   0.2ms divisors=   0.2ms stack=   0.2ms submit=   0.2ms


n=  9437184:   8972 cách | backtrack=  12.9ms divisors=   9.0ms stack=  15.4ms submit=  13.1ms


n=  5040000:  40773 cách | backtrack=  61.8ms divisors=  33.8ms stack=  46.6ms submit=  76.8ms
PASS n lớn: các cách khớp nhau
PASS n ~ 10^5: Cách có nhớ, oracle, nộp khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sinh hoán vị trùng (không giữ thứ tự không giảm)
Xem `sai_hoan_vi` ở phần test: `[2,6]` và `[6,2]` bị coi là hai cách. **Code đúng:** truyền `start = j` xuống đệ quy và chỉ thử thừa số `≥ start`.

### Lỗi 2: Đưa chính `n` (hoặc `1`) vào đáp án
#### Code sai

In [19]:
def sai_co_n(n):
    ans, path = [], []

    def dfs(m, i):
        ans.append(path + [m])                          # SAI: ghi cả khi path rỗng -> có [n]
        j = i
        while j * j <= m:
            if m % j == 0:
                path.append(j)
                dfs(m // j, j)
                path.pop()
            j += 1

    dfs(n, 2)
    return ans


print("n=12 đúng:", normalize(sol.getFactors(12)), "| sai:", normalize(sai_co_n(12)))
print("n=37 đúng:", sol.getFactors(37), "| sai:", sai_co_n(37))
print("n=1  đúng:", sol.getFactors(1), "| sai:", sai_co_n(1))

n=12 đúng: [[2, 2, 3], [2, 6], [3, 4]] | sai: [[2, 2, 3], [2, 6], [3, 4], [12]]
n=37 đúng: [] | sai: [[37]]
n=1  đúng: [] | sai: [[1]]


**Vì sao sai:** đề cấm thừa số `n` (phạm vi `[2, n-1]`), nên cách "`n = n`" không được ghi. Với số nguyên tố sẽ ra `[[37]]` thay vì `[]`, và `n = 1` ra `[[1]]`. **Code đúng:** chỉ ghi khi `path` không rỗng (`if path:`).

### Lỗi 3: Quên thừa số cuối (phần còn lại)
#### Code sai

In [20]:
def sai_quen_con_lai(n):
    ans, path = [], []

    def dfs(m, i):
        j = i
        while j * j <= m:
            if m % j == 0:
                path.append(j)
                if m // j == j:
                    ans.append(path[:])                  # SAI: chỉ ghi khi phần còn lại bằng j, bỏ quên m // j
                dfs(m // j, j)
                path.pop()
            j += 1

    dfs(n, 2)
    return ans


print("n=12 đúng:", normalize(sol.getFactors(12)), "| sai:", normalize(sai_quen_con_lai(12)))
print("n=16 đúng:", normalize(sol.getFactors(16)), "| sai:", normalize(sai_quen_con_lai(16)))

n=12 đúng: [[2, 2, 3], [2, 6], [3, 4]] | sai: []
n=16 đúng: [[2, 2, 2, 2], [2, 2, 4], [2, 8], [4, 4]] | sai: [[2, 2, 2], [4]]


**Vì sao sai:** `[2, 6]` gồm thừa số đã chọn `2` và **phần còn lại `6`** (một thừa số hợp lệ không cần tách tiếp). Phải ghi `path + [m]` (hoặc `path + [j, m // j]`).

### Lỗi 4: Không cắt tỉa `j*j ≤ m` (và ghi nhầm thừa số lớn hơn phần còn lại)
#### Code sai

In [21]:
def sai_khong_cat_tia(n):
    ans, path = [], []
    calls = 0

    def dfs(m, i):
        nonlocal calls
        calls += 1
        if path:
            ans.append(path + [m])
        for j in range(i, m):                           # SAI/chậm: quét tới m - 1
            if m % j == 0:
                path.append(j)
                dfs(m // j, j)                          # j > m // j khi j lớn: sinh thừa số không giảm bị vi phạm
                path.pop()

    dfs(n, 2)
    return ans, calls


res, calls = sai_khong_cat_tia(36)
print("n=36 sai có thể ghi cả bộ KHÔNG giảm:", [c for c in res if any(c[i] > c[i + 1] for i in range(len(c) - 1))][:4])
print("số lời gọi dfs của bản không cắt tỉa (n=36):", calls)

n=36 sai có thể ghi cả bộ KHÔNG giảm: [[2, 3, 3, 2], [2, 6, 3], [2, 9, 2], [3, 4, 3]]
số lời gọi dfs của bản không cắt tỉa (n=36): 17


**Vì sao sai:** nếu `j > m // j` thì sau khi chọn `j` phần còn lại `m // j < j` và bộ `path + [m // j]` **không giảm** bị phá vỡ (ví dụ ghi ra `[2, 9, 2]` cho `n = 36`). Điều kiện `j * j <= m` vừa đảm bảo **thứ tự không giảm** vừa **cắt tỉa** thời gian.

### Lỗi 5: Alias danh sách (`ans.append(path)`)
#### Code sai

In [22]:
def sai_alias(n):
    ans, path = [], []

    def dfs(m, i):
        if path:
            path.append(m)                              # SAI: thêm rồi lưu chính path
            ans.append(path)
            path.pop()
        j = i
        while j * j <= m:
            if m % j == 0:
                path.append(j)
                dfs(m // j, j)
                path.pop()
            j += 1

    dfs(n, 2)
    return ans


print("đúng:", normalize(sol.getFactors(12)), "| sai:", sai_alias(12))

đúng: [[2, 2, 3], [2, 6], [3, 4]] | sai: [[], [], []]


**Vì sao sai:** `ans` chứa **cùng một đối tượng** `path`; về cuối `path` rỗng nên toàn bộ phần tử đều thành `[]`. **Code đúng:** `ans.append(path + [m])` (danh sách mới) hoặc `path[:]`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**mọi cách viết một số thành tích / tổng / ghép**", "**không quan tâm thứ tự**" → **Backtracking với thứ tự không giảm** (truyền `start`).
- Với phân tích thành tích: chỉ cần thử thừa số nhỏ nhất `d` với **`d * d <= m`**; phần còn lại làm thừa số cuối hoặc đệ quy tiếp.
- Khung chung của họ **Combination Sum**: `chọn d → đệ quy(m / d hoặc m - d, d hoặc d + 1) → bỏ chọn`. Ở đây dùng `m // d` (tích) thay vì `m - d` (tổng).
- Nhớ điều kiện ghi nhận: **`if path:`** để không ghi cách "một thừa số bằng `n`".
- `n ≤ 10^7`: `√n ≈ 3162` ⇒ quét thẳng tới `√m` là đủ; không cần sàng nguyên tố.
- Số cách phân tích của `n` liên quan **hàm phân hoạch tích** (multiplicative partition): thử vài `n` nhỏ để kiểm tra (`12 → 3`, `16 → 4`, `30 → 4`, `36 → 8`).

## 15. Biến thể của bài

1. **Đếm** số cách phân tích (không liệt kê): DP/memo trên `(m, lo)`.
2. **Combination Sum** (39/40/216): chọn số để có **tổng** mục tiêu thay vì tích.
3. **Factor Combinations với `k` thừa số đúng**: chỉ giữ các bộ độ dài `k`.
4. **Số cách tối đa** trong khoảng `[1, N]`: duyệt từng `n` và đếm.
5. Cho phép **thừa số `1`**: cần giới hạn độ dài kẻo vô hạn.
#### Code demo

In [23]:
def count_factorizations(n):
    """Đếm số cách phân tích (bỏ cách (n,)) bằng DP có nhớ trên (m, lo)."""
    @lru_cache(maxsize=None)
    def g(m, lo):
        total = 1 if m >= lo else 0
        d = lo
        while d * d <= m:
            if m % d == 0:
                total += g(m // d, d)
            d += 1
        return total

    return max(0, g(n, 2) - 1) if n >= 4 else 0


def factors_with_k_parts(n, k):
    return [c for c in Solution().getFactors(n) if len(c) == k]


print([count_factorizations(n) for n in (12, 16, 30, 36, 37, 1)])      # [3, 4, 4, 8, 0, 0]
print(factors_with_k_parts(36, 3))                                       # các bộ gồm đúng 3 thừa số
assert count_factorizations(10 ** 5) == len(Solution().getFactors(10 ** 5))

[3, 4, 4, 8, 0, 0]
[[2, 2, 9], [2, 3, 6], [3, 3, 4]]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét `d` tới `m - 1` | thử mọi ước, giữ thứ tự không giảm | tầng đầu `O(n)` | `O(log n)` | chỉ làm baseline, `n` nhỏ |
| 2. Backtracking `d*d ≤ m` | chọn ước nhỏ nhất, đệ quy phần còn lại | `~O(√n + K·L)` | `O(log n)` | **cách nộp chuẩn** |
| 3. Trên danh sách ước | chỉ thử các ước của `n` | `O(√n + K·D)` | `O(D)` | `n` ít ước nhưng lớn |
| 4. Có nhớ | `f(m, lo)` trả mọi cách, nhớ kết quả | phụ thuộc đầu ra | lớn | cầu nối sang bài đếm / DP |
| 5. Stack tường minh | DFS không đệ quy | như Cách 2 | `O(L·√n)` | tránh đệ quy |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `Solution().getFactors(6)`; (b) `Solution().getFactors(4)`; (c) `[d for d in range(1, 21) if 20 % d == 0]`; (d) `len(Solution().getFactors(32))`.

**Bài 2 (tính bằng tay).** Liệt kê mọi cách phân tích của `n = 30` thành thừa số trong `[2, 29]` (không giảm).

**Bài 3 (điền TODO).** Hoàn thiện `factors_todo(n)` theo Cách 2.
#### Code demo

In [24]:
def factors_todo(n):
    ans, path = [], []

    def dfs(m, i):
        # TODO 1: nếu path không rỗng thì ans.append(path + [m])
        # TODO 2: j = i; while j * j <= m
        # TODO 3: nếu m % j == 0: path.append(j); dfs(m // j, j); path.pop()
        # TODO 4: tăng j
        pass

    # TODO 5: gọi dfs(n, 2) rồi trả ans
    pass


# Tự làm xong rồi mới mở test:
# for n, expected in examples:
#     assert normalize(factors_todo(n)) == normalize(expected)

**Bài 4 (tự viết lại).** Viết `factors_iter(n)` bằng stack tường minh (Cách 5), **không nhìn lại** code mẫu; kiểm tra với `oracle` cho `n = 1..300`.

**Bài 5 (biến thể / mini challenge).** Viết `max_parts(n)` trả về **số thừa số tối đa** trong một cách phân tích của `n` (với `n ≥ 2` là số nguyên tố mũ: `Ω(n)`, số thừa số nguyên tố tính cả bội) và `min_sum_factorization(n)` trả về cách phân tích có **tổng các thừa số nhỏ nhất** (với `n` nguyên tố thì chỉ có `[n]`). Thử `n = 36`: `max_parts = 4` (`2·2·3·3`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `4 = 2 × 2` có một cách; `6 = 2 × 3`; `32 = 2^5`. Bài 2: chọn thừa số nhỏ nhất `2`, `3`, `5`, nhớ thứ tự không giảm. Bài 5: số thừa số nhiều nhất khi tách thành các thừa số nguyên tố.
- **Mức 2 (vừa):** Bài 1: `[[2, 3]]`, `[[2, 2]]`, `[1, 2, 4, 5, 10, 20]`, `6` (`p(5) - 1 = 6`). Bài 2: `[2,15]`, `[3,10]`, `[5,6]`, `[2,3,5]`. Bài 3: khung Cách 2. Bài 5: `max_parts` = tổng các số mũ trong phân tích nguyên tố.
- **Mức 3 (sâu):** Bài 4: stack phần tử `(m, i, path)`; nhớ `path + (j,)`. Bài 5: `36 = 2²·3²` nên `max_parts = 4`; `min_sum_factorization(36)` là cách có tổng nhỏ nhất (`[6, 6]` có tổng `12`, `[2,2,3,3]` có tổng `10`) — hãy kiểm tra bằng code và thử tìm quy luật.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
# Bài 1
assert normalize(Solution().getFactors(6)) == [[2, 3]]
assert Solution().getFactors(4) == [[2, 2]]
assert [d for d in range(1, 21) if 20 % d == 0] == [1, 2, 4, 5, 10, 20]
assert len(Solution().getFactors(32)) == 6

# Bài 2
assert normalize(Solution().getFactors(30)) == normalize([[2, 15], [3, 10], [5, 6], [2, 3, 5]])


# Bài 3
def factors_todo_ans(n):
    ans, path = [], []

    def dfs(m, i):
        if path:
            ans.append(path + [m])
        j = i
        while j * j <= m:
            if m % j == 0:
                path.append(j)
                dfs(m // j, j)
                path.pop()
            j += 1

    dfs(n, 2)
    return ans


for n, expected in examples:
    assert normalize(factors_todo_ans(n)) == normalize(expected)


# Bài 4
def factors_iter(n):
    ans, stack = [], [(n, 2, ())]
    while stack:
        m, i, path = stack.pop()
        if path:
            ans.append(list(path) + [m])
        j = i
        while j * j <= m:
            if m % j == 0:
                stack.append((m // j, j, path + (j,)))
            j += 1
    return ans


for n in range(1, 301):
    assert normalize(factors_iter(n)) == oracle(n)


# Bài 5
def prime_exponents(n):
    exps, d = [], 2
    while d * d <= n:
        e = 0
        while n % d == 0:
            n //= d
            e += 1
        if e:
            exps.append(e)
        d += 1
    if n > 1:
        exps.append(1)
    return exps


def max_parts(n):
    return sum(prime_exponents(n)) if n >= 2 else 0


def min_sum_factorization(n):
    options = Solution().getFactors(n) + [[n]]
    return min(options, key=lambda c: (sum(c), c))


assert max_parts(36) == 4 and max_parts(37) == 1 and max_parts(12) == 3
assert min_sum_factorization(36) == [2, 2, 3, 3] and min_sum_factorization(37) == [37]
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (chọn ước nhỏ nhất, truyền `start`, cắt tỉa `j*j ≤ m`)

```python
class Solution:
    def getFactors(self, n: int) -> List[List[int]]:
        def dfs(m, i):
            if path:
                ans.append(path + [m])
            j = i
            while j * j <= m:
                if m % j == 0:
                    path.append(j)
                    dfs(m // j, j)
                    path.pop()
                j += 1

        path, ans = [], []
        dfs(n, 2)
        return ans
```

**Checklist**

- [ ] Thừa số **không giảm** (truyền `start = j`) để không sinh hoán vị trùng?
- [ ] Ghi nhận `path + [m]` chỉ khi `path` **không rỗng** (không có cách `[n]`; `n = 1` ra `[]`)?
- [ ] Cắt tỉa `j * j <= m` (không quét tới `m - 1`)?
- [ ] Lưu **bản sao** (`path + [m]`), không lưu chính `path`?
- [ ] Mỗi `append` đi kèm `pop`?
- [ ] Đã thử `n = 1`, số nguyên tố, `4`, `12`, `16`, `36`, `10^7`?
- [ ] Đã đối chiếu với brute force tách đôi `a × b` rồi sắp xếp?
- [ ] Skeleton đúng `class Solution` / `def getFactors(self, n: int) -> List[List[int]]`?